# Project 1: Initial Dairy Data Inspection

This notebook loads the private dairy dataset, checks its structure and quality, and summarizes missing cow IDs and milk-yield values. It does not modify the original file.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

## 1. Load the original data
The CSV should remain in `data/raw/` and is excluded from GitHub by `.gitignore`.

In [ ]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_path = project_root / "data" / "raw" / "project_1_ANSC_4040_dataset.csv"

if not data_path.exists():
    raise FileNotFoundError(
        f"Dataset not found at {data_path}. Place the private CSV there before continuing."
    )

df = pd.read_csv(data_path)
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
df.head()

## 2. Inspect columns and data types

In [ ]:
column_summary = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "non_missing": df.notna().sum(),
    "missing": df.isna().sum(),
    "missing_percent": df.isna().mean().mul(100).round(2),
    "unique_values": df.nunique(dropna=True),
})
column_summary.sort_values("missing_percent", ascending=False)

## 3. Check duplicates and descriptive statistics

In [ ]:
print(f"Exact duplicate rows: {df.duplicated().sum():,}")
df.describe(include="all").T

## 4. Visualize missingness

In [ ]:
missing_percent = df.isna().mean().mul(100).sort_values(ascending=False)
missing_percent = missing_percent[missing_percent > 0]

if missing_percent.empty:
    print("No missing values were detected.")
else:
    ax = missing_percent.plot(kind="bar", figsize=(10, 5), color="#2f6f4e")
    ax.set(title="Missing Values by Column", xlabel="Column", ylabel="Missing values (%)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 5. Identify the cow-ID and milk-yield columns
Update the two names below after reviewing `df.columns`. Keeping this choice explicit prevents the notebook from silently selecting the wrong variables.

In [ ]:
print(df.columns.tolist())

cow_id_column = "cow_id"          # Change to the actual column name
milk_yield_column = "milk_yield"  # Change to the actual column name

required_columns = {cow_id_column, milk_yield_column}
missing_required = required_columns - set(df.columns)
if missing_required:
    raise KeyError(f"Update the column-name variables. Not found: {sorted(missing_required)}")

In [ ]:
missing_pattern = pd.DataFrame({
    "cow_id_missing": df[cow_id_column].isna(),
    "milk_yield_missing": df[milk_yield_column].isna(),
})

pattern_counts = (
    missing_pattern.value_counts()
    .rename("row_count")
    .reset_index()
)
pattern_counts

## 6. Inspect cows and milk-yield values

In [ ]:
print(f"Known cows: {df[cow_id_column].nunique(dropna=True):,}")
display(df[cow_id_column].value_counts(dropna=False).head(20))
display(df[milk_yield_column].describe())

In [ ]:
yield_values = pd.to_numeric(df[milk_yield_column], errors="coerce")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(yield_values.dropna(), kde=True, ax=axes[0], color="#2f6f4e")
axes[0].set_title("Milk-Yield Distribution")
sns.boxplot(x=yield_values.dropna(), ax=axes[1], color="#8fc49f")
axes[1].set_title("Milk-Yield Range and Potential Outliers")
plt.tight_layout()
plt.show()

## 7. Questions to resolve before modeling

- What units are used for milk yield?
- Does a row represent a milking, a daily total, or another interval?
- Is there a timestamp, lactation number, days-in-milk value, milking station, or session field?
- Are blank IDs caused by tag-reader failure, data-entry error, or something else?
- Can one cow appear multiple times in a day?
- Are zero or negative yields possible, or are they error codes?
- Should the final model predict future missing values for known cows or generalize to cows it has never seen?

These answers determine the appropriate features, split strategy, and whether cow-ID recovery is defensible.